# 1.3. Estimación puntual e intervalos de confianza

In [1]:
SEMILLA = 42

from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import t

ruta_relativa = Path("data/raw/weatherAUS.csv")
ruta_csv = next(
    (carpeta / ruta_relativa for carpeta in (Path.cwd(), *Path.cwd().parents)
     if (carpeta / ruta_relativa).is_file()),
    None,
)
if ruta_csv is None:
    raise FileNotFoundError(ruta_relativa)

datos = pd.read_csv(ruta_csv)
print(f"Filas: {len(datos):,} | Columnas: {datos.shape[1]}".replace(",", "."))

Filas: 145.460 | Columnas: 23


## Método

Para cada variable se excluyen sus valores faltantes y se calcula la media muestral $\bar{x}$ de sus $n$ valores válidos. Cada fila válida corresponde a una observación diaria de una estación y tiene el mismo peso en la media. Por ello, las estaciones con más observaciones válidas aportan más al resultado.

La desviación estándar muestral $s$ usa $n-1$ grados de libertad. El error estándar es $s/\sqrt{n}$ y el intervalo bilateral al 95 % se obtiene como $\bar{x} \pm t_{0.975,\,n-1}\,s/\sqrt{n}$. El cálculo es determinista.

In [2]:
unidades = {"MaxTemp": "°C", "Humidity3pm": "%", "Pressure3pm": "hPa"}
registros = []

for variable, unidad in unidades.items():
    valores = datos[variable].dropna()
    n_valido = valores.size
    media = valores.mean()
    desviacion = valores.std(ddof=1)
    error = desviacion / n_valido**0.5
    margen = t.ppf(0.975, df=n_valido - 1) * error
    registros.append({
        "Variable": variable,
        "Unidad": unidad,
        "N válido": n_valido,
        "N faltante": len(datos) - n_valido,
        "Media": media,
        "Desviación estándar": desviacion,
        "Error estándar": error,
        "Límite inferior": media - margen,
        "Límite superior": media + margen,
    })

resultados = pd.DataFrame(registros)
display(resultados.round(3))

,Variable,Unidad,N válido,N faltante,Media,Desviación estándar,Error estándar,Límite inferior,Límite superior
0,MaxTemp,°C,144199,1261,23.221,7.119,0.019,23.185,23.258
1,Humidity3pm,%,140953,4507,51.539,20.796,0.055,51.431,51.648
2,Pressure3pm,hPa,130432,15028,1015.256,7.037,0.019,1015.218,1015.294


## Interpretación

Bajo los supuestos del intervalo t, si se repitiera el muestreo y se construyera un intervalo del mismo modo cada vez, aproximadamente el 95 % de esos intervalos contendría la media poblacional correspondiente.

In [3]:
mediciones = {
    "MaxTemp": "temperatura máxima diaria",
    "Humidity3pm": "humedad relativa a las 15:00",
    "Pressure3pm": "presión a las 15:00",
}
interpretaciones = []
for fila in registros:
    nombre = mediciones[fila["Variable"]]
    unidad = fila["Unidad"]
    media, inferior, superior = (
        f"{fila[columna]:.3f}".replace(".", ",")
        for columna in ("Media", "Límite inferior", "Límite superior")
    )
    interpretaciones.append(
        f"- La media estimada de **{nombre}** es **{media} {unidad}**. "
        f"El IC del 95 % para su media poblacional va de **{inferior} a {superior} {unidad}**."
    )
display(Markdown("\n".join(interpretaciones)))

- La media estimada de **temperatura máxima diaria** es **23,221 °C**. El IC del 95 % para su media poblacional va de **23,185 a 23,258 °C**.
- La media estimada de **humedad relativa a las 15:00** es **51,539 %**. El IC del 95 % para su media poblacional va de **51,431 a 51,648 %**.
- La media estimada de **presión a las 15:00** es **1015,256 hPa**. El IC del 95 % para su media poblacional va de **1015,218 a 1015,294 hPa**.

## Limitaciones

Las observaciones de una misma estación, especialmente las de días cercanos, pueden estar correlacionadas. También puede haber dependencia entre estaciones cercanas. La correlación positiva reduce la cantidad efectiva de información y puede hacer que los intervalos calculados bajo independencia sean demasiado estrechos.

Las tres estimaciones usan conjuntos de filas que pueden diferir. Si la disponibilidad de una medición depende de la estación, la temporada o las condiciones meteorológicas, las observaciones válidas pueden representar de forma desigual las condiciones del conjunto completo y desplazar las medias estimadas.